# LAFEX: fine-tune Whisper dengan data kontributor
**Sebelum mulai (jujur):** Whisper tidak punya Tetun. Hasil yang layak butuh puluhan jam audio bertranskripsi dari banyak penutur.
Dengan data sedikit, notebook ini hanya membuktikan alurnya bekerja. Skrip ini belum pernah dijalankan di Colab/Kaggle oleh pembuatnya: jalankan dulu dengan `--max-steps 20`.

Runtime: **GPU (T4)** — menu *Runtime > Change runtime type*.

In [ ]:
!pip install -q -U transformers accelerate soundfile peft

## 1. Unggah data
Di server LAFEX: `python manage.py export_suara --out data_suara --bahasa fataluku`, lalu zip folder `data_suara/whisper` menjadi `whisper.zip`.
Unggah `whisper.zip` dan `finetune_whisper.py` (dari folder `training/`).

In [ ]:
from google.colab import files
files.upload()          # pilih whisper.zip DAN finetune_whisper.py
!unzip -q -o whisper.zip -d data
!ls data | head

## 2. Uji cepat (20 langkah) — pastikan semuanya jalan

In [ ]:
!python finetune_whisper.py --data data/whisper --out hasil/uji --model openai/whisper-small --max-steps 20 --paksa

## 3. Latih sungguhan
- `whisper-small` muat di T4 gratis. Untuk `whisper-medium` tambahkan `--lora`.
- `--baseline` menghitung WER model asli sebagai pembanding.

In [ ]:
!python finetune_whisper.py --data data/whisper --out hasil/whisper-tet --model openai/whisper-small --epochs 8 --baseline

## 4. Konversi untuk server (faster-whisper) dan unduh
Setelah itu di `.env` LAFEX: `STT_URL=http://servermu:8000/v1` dan `STT_MODEL=<nama model>` (ikuti dokumentasi faster-whisper-server).

In [ ]:
!pip install -q ctranslate2
!ct2-transformers-converter --model hasil/whisper-tet --output_dir hasil/whisper-tet-ct2 --quantization float16 --copy_files tokenizer.json preprocessor_config.json
!zip -q -r whisper-tet-ct2.zip hasil/whisper-tet-ct2
files.download('whisper-tet-ct2.zip')